# Documentación del Proceso ETL: Consolidación Histórica y Enriquecimiento Espacial del Valor de Referencia

Este cuaderno contiene el proceso completo de extracción, transformación y carga (ETL) diseñado para estructurar la base de datos del proyecto. A lo largo de este flujo de trabajo, unificaremos, depuraremos y conectaremos múltiples fuentes de información urbana, inmobiliaria, turística y de seguridad de la ciudad.

El propósito central es construir una matriz territorial integrada a nivel de barrio y localidad, la cual transformará los datos dispersos en un insumo limpio, coherente y listo para alimentar los modelos de análisis espacial y de agrupamiento que darán forma a nuestro sistema de soporte a decisiones.

# Librerias

In [112]:
import os
import re
import glob
import zipfile
import warnings
import unicodedata
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Configuración de estilos para visualizaciones del informe
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10})

# 0. Extracción de datos

Se procede a realizar la extracción automática de los archivos comprimidos almacenados en el directorio principal de datos (data/).

Para garantizar un flujo de trabajo ordenado y evitar la manipulación manual de archivos, el proceso clasifica las fuentes según su nombre y contenido, creando una carpeta para cada fuente y dejarlas en un formato procesable.

In [113]:
os.makedirs('data/data_historica', exist_ok=True)
for f in glob.glob('data/*.zip'):
    if 'sector' not in f.lower() and 'transmilenio' not in f.lower() and 'manzana' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data/data_historica', os.path.splitext(os.path.basename(f))[0]))

os.makedirs('data/temp_sector', exist_ok=True)
for f in glob.glob('data/*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('data/temp_sector')

os.makedirs('data/temp_transmilenio', exist_ok=True)
for f in glob.glob('data/*transmilenio*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('data/temp_transmilenio')

os.makedirs('data/temp_manzana_estratificacion', exist_ok=True)
for f in glob.glob('data/*estrat*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('data/temp_manzana_estratificacion')

# 1. Unificación Localidad - Barrio - Geometria

Se unifica la información espacial de sectores catastrales y localidades para construir un diccionario común que sirve como base para integrar el resto de fuentes de información, adicionalmente se crea la columna area_km2 que contiene la superficie métrica de cada barrio proyectada en el sistema oficial, y la columna localidad para vincular cada sector con su respectiva división administrativa.

In [114]:
archivo_sector = glob.glob('data/temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)

gdf_localidad = gpd.read_file(glob.glob('data/loca*.gpkg')[0])
gdf_localidad = gdf_localidad.to_crs(gdf_sector.crs)

gdf_sector = gdf_sector.rename(columns={'SCANOMBRE': 'barrio'})

# Calcular el área en km² (Proyectando a MAGNA-SIRGAS Origen Nacional EPSG:9377 para precisión métrica)
gdf_sector_proyectado = gdf_sector.to_crs('EPSG:9377')
gdf_sector['area_km2'] = gdf_sector_proyectado.geometry.area / 1_000_000

# Definir qué columnas conservar de sector
cols_sector = ['barrio', 'area_km2', 'geometry']

gdf_barrios_base = gpd.sjoin(gdf_sector[cols_sector], gdf_localidad[['LocNombre', 'geometry']], how='left', predicate='intersects')
gdf_barrios_base = gdf_barrios_base.rename(columns={'LocNombre': 'localidad'})
gdf_barrios_base = gdf_barrios_base.drop(columns=['index_right'])

# Deduplicar polígonos de barrio
gdf_barrios_base = gdf_barrios_base.drop_duplicates(subset=['barrio']).reset_index(drop=True)

print(f"\nTamaño del Diccionario Espacial Base (Barrios): {gdf_barrios_base.shape[0]} filas, {gdf_barrios_base.shape[1]} columnas")


Tamaño del Diccionario Espacial Base (Barrios): 1150 filas, 4 columnas


In [115]:
gdf_barrios_base.head(5)

,barrio,area_km2,geometry,localidad
0,EL BOSQUE,0.055879,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525...",USME
1,BILBAO,0.295314,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ...",SUBA
2,CIUDAD KENNEDY CENTRAL,0.476473,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657...",KENNEDY
3,CONCEPCION NORTE,0.140259,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727...",CHAPINERO
4,COLON,0.285947,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043...",PUENTE ARANDA


## 2. Agregar estratificación

Se procesa la información geoespacial de la estratificación urbana a nivel de manzana para asignarle un estrato socioeconómico predominante a cada barrio. Para lograrlo, el script alinea los sistemas de referencia espacial, convierte los polígonos de las manzanas en puntos centrales (centroides) para evitar conflictos en las fronteras, y realiza un cruce espacial mediante una unión de superposición. Finalmente, agrupa los datos por barrio y calcula la moda estadística —el estrato más frecuente entre las manzanas que lo componen—, consolidando una nueva columna llamada estrato_predominante.

In [116]:
# 1. Leer el shapefile extraido de estratos
archivo_estrato = glob.glob('data/temp_manzana_estratificacion/**/*.shp', recursive=True)[0]
gdf_estrato = gpd.read_file(archivo_estrato)

print(f"Manzanas con estrato cargadas: {gdf_estrato.shape[0]}")

# 2. Alinear CRS con los barrios
if gdf_estrato.crs != gdf_barrios_base.crs:
    gdf_estrato = gdf_estrato.to_crs(gdf_barrios_base.crs)

# 3. Identificar la columna de estrato dinámicamente 
# (Suele llamarse 'ESTRATO', 'SCAESTRATO', 'estrato')
col_estrato = [c for c in gdf_estrato.columns if 'ESTRATO' in c.upper()][0]

# 4. Convertir manzanas a puntos (centroides) para evitar duplicidades en fronteras
gdf_estrato_centroides = gdf_estrato.copy()
gdf_estrato_centroides['geometry'] = gdf_estrato_centroides.geometry.centroid

# 5. Cruce espacial: ¿En qué barrio cae el centroide de cada manzana?
manzanas_por_barrio = gpd.sjoin(gdf_estrato_centroides, gdf_barrios_base[['barrio', 'geometry']], how='inner', predicate='intersects')

# 6. Agrupación: Calcular la MODA (el estrato más frecuente) por barrio
estrato_por_barrio = manzanas_por_barrio.groupby('barrio')[col_estrato].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan).reset_index()
estrato_por_barrio = estrato_por_barrio.rename(columns={col_estrato: 'estrato_predominante'})
print(f"Estrato predominante calculado para {estrato_por_barrio.shape[0]} barrios")

Manzanas con estrato cargadas: 44260
Estrato predominante calculado para 1042 barrios


In [117]:
estrato_por_barrio.head()

,barrio,estrato_predominante
0,ABRAHAM LINCOLN,2
1,ACACIAS USAQUEN,4
2,ACEVEDO TEJADA,3
3,AEROPUERTO EL DORADO,3
4,AGUAS CLARAS,0


# 3. Consolidación valores historicos de metro cuadrado

Se consolida toda la serie histórica de valor catastral recorriendo los archivos geográficos de los diferentes años. El script estandariza el sistema de coordenadas, extrae de forma automática el año correspondiente a partir del nombre del archivo y unifica las columnas clave de código de manzana y valor de referencia (v_ref). Posteriormente, mediante un cruce espacial, agrupa y asigna las manzanas a su respectivo barrio y localidad, calculando finalmente la mediana del valor comercial de referencia por barrio para cada año, lo que da origen a la matriz histórica resumida (df_vhi_barrio).

In [118]:
lista_gdfs = []
target_crs = "EPSG:4326"
 
for shp in glob.glob('data/data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
   
    # Asegurar que todas las capas usen el mismo CRS para poder concatenar
    if gdf.crs is None:
        gdf.set_crs(target_crs, inplace=True)
    elif gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
       
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=target_crs))
 
gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
print(f"Tamaño de Registros de Manzanas Históricas (gdf_maestro): {gdf_maestro.shape[0]} filas, {gdf_maestro.shape[1]} columnas")
 
gdf_barrios_base = gdf_barrios_base.to_crs(gdf_maestro.crs)
 
# Cruce espacial de manzanas con diccionario de barrios
gdf_historico = gpd.sjoin(gdf_maestro, gdf_barrios_base[['barrio', 'localidad', 'geometry']], how='inner', predicate='intersects')
if 'index_right' in gdf_historico.columns:
    gdf_historico = gdf_historico.drop(columns=['index_right'])
 
# Mediana del valor de referencia por barrio y año
df_vhi_barrio = gdf_historico.groupby(['localidad', 'barrio', 'ano'])['v_ref'].median().reset_index()
print(f"Tamaño del Consolidado VHI por Barrio y Año: {df_vhi_barrio.shape[0]} filas, {df_vhi_barrio.shape[1]} columnas")

Tamaño de Registros de Manzanas Históricas (gdf_maestro): 518935 filas, 4 columnas
Tamaño del Consolidado VHI por Barrio y Año: 13147 filas, 4 columnas


In [119]:
df_vhi_barrio.head(5)

,localidad,barrio,ano,v_ref
0,ANTONIO NARIÑO,CARACAS,2015,1300000.0
1,ANTONIO NARIÑO,CARACAS,2016,1400000.0
2,ANTONIO NARIÑO,CARACAS,2017,1700000.0
3,ANTONIO NARIÑO,CARACAS,2018,1700000.0
4,ANTONIO NARIÑO,CARACAS,2019,1750000.0


# 4. Extracción datos transmilenio

Se procesa la información geoespacial de las estaciones del sistema de transporte masivo para integrarla como una variable de control y accesibilidad urbana. El script lee la capa vectorial de Transmilenio, alinea su sistema de referencia de coordenadas con el diccionario base de barrios y realiza un cruce espacial por intersección. Finalmente, agrupa los resultados para contar cuántas estaciones operativas se ubican dentro de cada sector catastral, generando la columna num_estaciones_transmilenio.

In [120]:
archivo_transmilenio = glob.glob('data/temp_transmilenio/**/*.shp', recursive=True)[0]
gdf_transmilenio = gpd.read_file(archivo_transmilenio)
print(f"Estaciones de Transmilenio leídas: {gdf_transmilenio.shape[0]} filas, {gdf_transmilenio.shape[1]} columnas")

if gdf_barrios_base.crs != gdf_transmilenio.crs:
    gdf_transmilenio = gdf_transmilenio.to_crs(gdf_barrios_base.crs)

# Conteo espacial de estaciones por barrio
estaciones_por_barrio = gpd.sjoin(gdf_transmilenio, gdf_barrios_base[['barrio', 'geometry']], how='inner', predicate='intersects')
conteo_estaciones = estaciones_por_barrio.groupby('barrio').size().reset_index(name='num_estaciones_transmilenio')                                       
print(f"Barrios con al menos una estación de Transmilenio: {conteo_estaciones.shape[0]} barrios")                                                                                 

Estaciones de Transmilenio leídas: 153 filas, 28 columnas
Barrios con al menos una estación de Transmilenio: 111 barrios


In [121]:
conteo_estaciones.head(5)

,barrio,num_estaciones_transmilenio
0,ABRAHAM LINCOLN,1
1,ACEVEDO TEJADA,2
2,ALTOS DE CHOZICA,1
3,AUTOPISTA MUZU,1
4,AUTOPISTA MUZU ORIENTAL,1


# 5. Extracción datos delitos

Se procesa y depura la base de datos de seguridad para calcular una tasa de crecimiento anualizada de la criminalidad por localidad. El script carga los registros delictivos, limpia los nombres de columnas y elimina metadatos operativos innecesarios. Posteriormente, estandariza los campos de texto convirtiéndolos a mayúsculas y eliminando tildes y caracteres especiales, asegura el formato numérico correcto para los periodos y cantidades, y agrupa la información por año y localidad. A partir de esta tabla resumida, el script identifica de manera dinámica el primer y último año disponible en la serie para calcular el CAGR de delitos, depurando finalmente los valores infinitos o nulos para consolidar la matriz final de análisis.

In [122]:
delitos = pd.read_csv("data/delitos.csv", sep=";", low_memory=False)
delitos.columns = delitos.columns.str.strip()

delitos = delitos.drop(
    columns=[c for c in delitos.columns if c.strip().lower() in ["fecha de consulta", "datos actualizados hasta"]],
    errors="ignore",
)

columnas_texto = [ "SEXO", "HECHO", "LOCALIDAD", "ARMA_EMPLEADA", "RANGO_VITAL", "RANGO_DEL_DIA",]
for col in columnas_texto:
  if col in delitos.columns:
    delitos[col] = (delitos[col].astype("string").str.strip().str.upper().replace("", pd.NA))

for col in ["ANIO", "CANTIDAD"]:
  if col in delitos.columns:
    delitos[col] = pd.to_numeric(delitos[col], errors="coerce")

def quitar_tildes(texto):
  if pd.isna(texto):
    return texto
  texto = str(texto).upper().replace("Ñ", "N_TEMP")
  texto = "".join(c for c in unicodedata.normalize("NFD", texto)  if unicodedata.category(c) != "Mn")
  return texto.replace("N_TEMP", "Ñ").strip()

delitos["LOCALIDAD"] = delitos["LOCALIDAD"].apply(quitar_tildes)
delitos_pivot = (delitos.groupby(["ANIO", "LOCALIDAD"], dropna=False)["CANTIDAD"].sum().reset_index()
    .pivot(index="LOCALIDAD", columns="ANIO", values="CANTIDAD")
)

anios_disponibles = sorted([col for col in delitos_pivot.columns if isinstance(col, (int, float))])
col_inicial, col_final = anios_disponibles[0], anios_disponibles[-1]
n_periodos = float(col_final - col_inicial)

delitos_pivot["cagr_delitos_18_26"] = ((delitos_pivot[col_final] / delitos_pivot[col_inicial]) ** (1.0 / n_periodos) - 1.0)
delitos_pivot = delitos_pivot.rename(columns={col_inicial: f"delitos_{int(col_inicial)}", col_final: f"delitos_{int(col_final)}"})

In [123]:
delitos_pivot

ANIO,delitos_2018,2019,2020,2021,2022,2023,2024,2025,delitos_2026,cagr_delitos_18_26
LOCALIDAD,,,,,,,,,,
ANTONIO NARIÑO,3874.0,4459.0,3556.0,4287.0,4454.0,4794.0,3837.0,3717.0,2926.0,-0.034473
BARRIOS UNIDOS,7267.0,7999.0,5568.0,6829.0,8000.0,9277.0,7812.0,7437.0,6209.0,-0.019476
BOSA,14271.0,14796.0,13295.0,15014.0,14607.0,15127.0,12641.0,12840.0,11690.0,-0.024629
CANDELARIA,2260.0,3084.0,1730.0,2307.0,2578.0,1907.0,2719.0,2266.0,1616.0,-0.041060
CHAPINERO,13995.0,17228.0,7915.0,10946.0,14967.0,17954.0,14408.0,12417.0,9476.0,-0.047573
CIUDAD BOLIVAR,12005.0,12961.0,11525.0,13198.0,13866.0,17777.0,14890.0,14828.0,11076.0,-0.010017
ENGATIVA,19406.0,20916.0,17663.0,20411.0,22148.0,23827.0,21114.0,21556.0,16299.0,-0.021574
FONTIBON,11030.0,12258.0,9101.0,10720.0,13532.0,12960.0,13043.0,12368.0,9562.0,-0.017694
KENNEDY,24725.0,26895.0,20976.0,23985.0,25787.0,27243.0,23932.0,22319.0,17188.0,-0.044433


In [124]:
cols_a_conservar = [f"delitos_{int(col_inicial)}", f"delitos_{int(col_final)}", "cagr_delitos_18_26"]
df_cagr_delitos_final = (delitos_pivot[cols_a_conservar].replace([np.inf, -np.inf], np.nan).dropna(subset=["cagr_delitos_18_26"]).reset_index())
df_cagr_delitos_final = df_cagr_delitos_final.rename(columns={"LOCALIDAD": "localidad"})

In [125]:
df_cagr_delitos_final.head(5)

ANIO,localidad,delitos_2018,delitos_2026,cagr_delitos_18_26
0,ANTONIO NARIÑO,3874.0,2926.0,-0.034473
1,BARRIOS UNIDOS,7267.0,6209.0,-0.019476
2,BOSA,14271.0,11690.0,-0.024629
3,CANDELARIA,2260.0,1616.0,-0.041060
4,CHAPINERO,13995.0,9476.0,-0.047573


# 6. Cálculo CAGR 2015-2026

Se estructuran y auditan las series históricas del valor comercial del suelo a nivel de barrio para el periodo comprendido entre 2015 y 2026. El script transforma los datos anuales en una matriz ancha (pivot) donde cada columna representa el valor de referencia de un año específico.

In [126]:
df_cagr_base = df_vhi_barrio[df_vhi_barrio['ano'].isin([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])]
df_pivot = df_cagr_base.pivot(index=['localidad', 'barrio'], columns='ano', values='v_ref').reset_index()
df_pivot = df_pivot.rename(columns={2015: 'v_ref_2015', 2016: 'v_ref_2016', 2017: 'v_ref_2017', 
                                    2018: 'v_ref_2018', 2019: 'v_ref_2019', 2020: 'v_ref_2020', 
                                    2021: 'v_ref_2021', 2022: 'v_ref_2022', 2023: 'v_ref_2023', 
                                    2024: 'v_ref_2024', 2025: 'v_ref_2025', 2026: 'v_ref_2026'})

def auditar_datos_inmobiliarios(df):
    """
    Función para auditar el estado de los valores de referencia (2015, 2026).
    Muestra conteo de nulos, registros afectados y desglose específico de anomalías.
    """

    cols_eval = ['v_ref_2015', 'v_ref_2026']

    print("\n1. Conteo de valores nulos (NaN) por columna:")
    print(df[cols_eval].isnull().sum())
    print("\n2. Conteo de valores en cero (0) por columna:")
    print((df[cols_eval] == 0).sum())
    print("\n3. Desglose detallado de anomalías (NaN o 0) por año:")

    for col in cols_eval:
        n_nans = df[col].isnull().sum()
        n_zeros = (df[col] == 0).sum()
        print(f"   - {col}: {n_nans} NaNs | {n_zeros} ceros (Total anomalías: {n_nans + n_zeros})")
    
    condicion_critica = (
        df['v_ref_2015'].isnull() | (df['v_ref_2015'] == 0) |
        df['v_ref_2026'].isnull() | (df['v_ref_2026'] == 0) 
    )
    filas_criticas = df[condicion_critica]
    print(f"\n4. Total de registros afectados con anomalías entre 2015 - 2026: {len(filas_criticas)}")
    
    if len(filas_criticas) > 0:
        print("\n5. Muestra de registros con anomalías críticas:")
        display(filas_criticas[['localidad', 'barrio', 'v_ref_2015', 'v_ref_2026']].head(10))
    else:
        print("\n5. ¡No se encontraron registros críticos entre 2015 - 2026!")

auditar_datos_inmobiliarios(df_pivot)


1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015     0
v_ref_2026    13
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    68
v_ref_2026     0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 68 ceros (Total anomalías: 68)
   - v_ref_2026: 13 NaNs | 0 ceros (Total anomalías: 13)

4. Total de registros afectados con anomalías entre 2015 - 2026: 76

5. Muestra de registros con anomalías críticas:


ano,localidad,barrio,v_ref_2015,v_ref_2026
35,BOSA,BOSA 37,0.0,1391520.0
40,BOSA,CAMPO VERDE,0.0,1498952.0
42,BOSA,CANAVERALEJO RURAL,0.0,38100.0
45,BOSA,CAÑAVERALEJO I,0.0,38100.0
56,BOSA,EL CORZO II,0.0,38100.0
60,BOSA,EL REMANSO,0.0,162600.0
97,BOSA,SAN BERNARDINO XIX,0.0,1525000.0
98,BOSA,SAN BERNARDINO XVIII,0.0,162100.0
99,BOSA,SAN BERNARDINO XXII URBANO,0.0,821500.0
135,CHAPINERO,INGEMAR ORIENTAL I,0.0,133400.0


Posteriormente, se ejecuta una función de auditoría técnica para cuantificar la presencia de valores nulos o ceros que representan errores de captura. Para corregirlos sin sesgar la distribución espacial, se aplica un proceso de limpieza e imputación en varias fases: primero se normalizan los ceros lógicamente imposibles a nulos; luego, se imputa el año base (2015) utilizando la mediana zonal por localidad para preservar la tendencia macroespacial; y para el año objetivo (2026), se implementa una estrategia combinada que prioriza una interpolación lineal basada en la trayectoria histórica de cada barrio, respaldada con la mediana de la localidad en caso de persistir vacíos.

In [127]:
anios_historicos = [f'v_ref_{anio}' for anio in range(2015, 2027)]

# A. Normalización de ceros a NaN: 
# Los ceros en precios inmobiliarios por metro cuadrado corresponden a vacíos de información,
# no a valor comercial real, por lo que se estandarizan a NaN para evitar sesgos matemáticos.
df_pivot[anios_historicos] = df_pivot[anios_historicos].replace(0, np.nan)

# B. Imputación 2015 (Corte Base):
# Se aplica la mediana del valor de referencia agrupada por localidad. 
# Justificación: Preserva la tendencia macroespacial zonal sin distorsionar con valores extremos (outliers).
df_pivot['v_ref_2015'] = df_pivot['v_ref_2015'].fillna(df_pivot.groupby('localidad')['v_ref_2015'].transform('median'))

# C. Imputación 2026 (Trayectoria Histórica + Respaldo Zonal):
# 1. Primero se intenta rellenar usando interpolación lineal con los años intermedios disponibles por cada fila,
#    respetando la inercia de precios particular de cada barrio.
df_pivot['v_ref_2026'] = df_pivot['v_ref_2026'].fillna(df_pivot[anios_historicos].interpolate(method='linear', axis=1)['v_ref_2026'])
# 2. Si aún persisten vacíos (barrios sin ningún histórico intermedio), se aplica la mediana de la localidad como respaldo.
df_pivot['v_ref_2026'] = df_pivot['v_ref_2026'].fillna(df_pivot.groupby('localidad')['v_ref_2026'].transform('median'))

auditar_datos_inmobiliarios(df_pivot)


1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015    5
v_ref_2026    0
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 5 NaNs | 0 ceros (Total anomalías: 5)
   - v_ref_2026: 0 NaNs | 0 ceros (Total anomalías: 0)

4. Total de registros afectados con anomalías entre 2015 - 2026: 5

5. Muestra de registros con anomalías críticas:


ano,localidad,barrio,v_ref_2015,v_ref_2026
838,SUMAPAZ,LA UNION URBANO,NaN,39390.0
839,SUMAPAZ,NAZARETH,NaN,35000.0
840,SUMAPAZ,NAZARETH URBANO,NaN,40000.0
841,SUMAPAZ,SAN JUAN,NaN,37500.0
842,SUMAPAZ,SAN JUAN URBANO,NaN,37500.0


Finalmente, se eliminan los registros huérfanos o irrecuperables que no cuentan con la información base suficiente y se calcula la Tasa de Crecimiento Anual Compuesta (CAGR) para los 11 años transcurridos entre 2015 y 2026 (cagr_15_26), consolidando la métrica central de aceleración inmobiliaria para el modelo.

In [128]:
# 1. Filtrar el DataFrame para eliminar aquellas filas que siguen teniendo 
# anomalías críticas (NaN o ceros) en 2015 o 2026 después de la imputación.
# (Esto removerá exactamente esos 5 registros que no pudieron ser imputados).
df_pivot = df_pivot[
    ~(
        df_pivot['v_ref_2015'].isnull() | (df_pivot['v_ref_2015'] == 0) |
        df_pivot['v_ref_2026'].isnull() | (df_pivot['v_ref_2026'] == 0)
    )
].copy()

print(f"Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: {df_pivot.shape}")
auditar_datos_inmobiliarios(df_pivot)

Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: (1098, 14)

1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015    0
v_ref_2026    0
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 0 NaNs | 0 ceros (Total anomalías: 0)

4. Total de registros afectados con anomalías entre 2015 - 2026: 0

5. ¡No se encontraron registros críticos entre 2015 - 2026!


In [129]:
# Calcular CAGR de 12 años (2015 a 2026)
n_anos = 2026 - 2015
df_pivot['cagr_15_26'] = (df_pivot['v_ref_2026'] / df_pivot['v_ref_2015']) ** (1 / n_anos) - 1
print(f"Tamaño de la tabla de CAGR (2015-2026): {df_pivot.shape[0]} filas, {df_pivot.shape[1]} columnas")

Tamaño de la tabla de CAGR (2015-2026): 1098 filas, 15 columnas


In [130]:
df_pivot.head(5)

ano,localidad,barrio,v_ref_2015,v_ref_2016,v_ref_2017,v_ref_2018,v_ref_2019,v_ref_2020,v_ref_2021,v_ref_2022,v_ref_2023,v_ref_2024,v_ref_2025,v_ref_2026,cagr_15_26
0,ANTONIO NARIÑO,CARACAS,1300000.0,1400000.0,1700000.0,1700000.0,1750000.0,1900000.0,2200000.0,2200000.0,2500000.0,2570000.0,2679000.0,3200000.0,0.085336
1,ANTONIO NARIÑO,EDUARDO SANTOS,900000.0,1100000.0,1200000.0,1600000.0,1800000.0,2000000.0,2350000.0,2350000.0,2560000.0,2800000.0,2950000.0,3000000.0,0.115667
2,ANTONIO NARIÑO,EL VERGEL,912500.0,1037500.0,1250000.0,1625000.0,1800000.0,2000000.0,2350000.0,2350000.0,2560000.0,2750000.0,2900000.0,3000000.0,0.114268
3,ANTONIO NARIÑO,LA FRAGUITA,1000000.0,1200000.0,1350000.0,1600000.0,1800000.0,2000000.0,2350000.0,2350000.0,2560000.0,2750000.0,2900000.0,3000000.0,0.105032
4,ANTONIO NARIÑO,SANTANDER,1000000.0,1200000.0,1250000.0,1600000.0,2000000.0,2100000.0,2350000.0,2350000.0,2650000.0,2820000.0,3040000.0,3150000.0,0.109944


# 7. Cargar listings agrupados

Se cargan y unifican los archivos procesados de los listados de alojamientos turísticos y de delitos utilizando una búsqueda recursiva en los directorios de trabajo. El script localiza automáticamente los archivos CSV correspondientes tanto a Inside Airbnb como a la base de delincuencia previamente limpia, reportando el volumen de registros cargados para cada caso. Finalmente, se renombra la columna de conteo de alojamientos en la tabla de listados como num_listings_airbnb para estandarizar su nomenclatura de cara al cruce y modelo final.

In [131]:
archivo_listings = (glob.glob('data/**/*listings*.csv', recursive=True) + glob.glob('temp_listings/**/*listings*.csv', recursive=True))[0]
listings = pd.read_csv(archivo_listings)
print(f"Listings cargados desde {archivo_listings}: {len(listings)} filas")

archivo_delitos = (glob.glob('data/**/*delitos*.csv', recursive=True) + glob.glob('temp_delitos/**/*delitos*.csv', recursive=True))[0]
delitos = pd.read_csv(archivo_delitos)
print(f"Delitos cargados desde {archivo_delitos}: {len(delitos)} filas")

listings = listings.rename(columns={'conteo': 'num_listings_airbnb'})

Listings cargados desde data\listings_agrupado_barrio_localidad.csv: 654 filas
Delitos cargados desde data\delitos.csv: 376185 filas


In [132]:
listings.head(5)

,localidad,barrio,num_listings_airbnb
0,ANTONIO NARIÑO,CIUDAD BERNA,1
1,ANTONIO NARIÑO,CIUDAD JARDIN SUR,4
2,ANTONIO NARIÑO,EDUARDO FREY,5
3,ANTONIO NARIÑO,LA FRAGUA,1
4,ANTONIO NARIÑO,LA FRAGUITA,7


# 8. Unificación datos

Se consolida la matriz final de datos integrando todas las dimensiones espaciales, económicas y urbanas en el diccionario base de barrios. El script realiza cruces sucesivos para incorporar los valores de referencia y el CAGR inmobiliario, el macrofactor de criminalidad por localidad, el estrato socioeconómico predominante (imputando valores faltantes mediante la moda zonal), la infraestructura de transporte masivo (Transmilenio) y la dinámica de alojamiento turístico (Airbnb), transformando los conteos absolutos de esta última en una métrica normalizada de densidad espacial (densidad_airbnb_km2).

In [133]:
matriz_final = gdf_barrios_base.copy()

# Fusión con CAGR y valores de referencia
matriz_final = matriz_final.merge(df_pivot[['barrio', 'localidad', 'v_ref_2015', 'v_ref_2026', 'cagr_15_26']], on=['barrio', 'localidad'], how='left')

# Fusión con CAGR de delitos por localidad
matriz_final = matriz_final.merge(df_cagr_delitos_final[["localidad", "delitos_2018", "delitos_2026", "cagr_delitos_18_26"]], on="localidad", how="left")

# Fusión con Estrato Predominante
matriz_final = matriz_final.merge(estrato_por_barrio, on='barrio', how='left')
matriz_final['estrato_predominante'] = matriz_final['estrato_predominante'].fillna(
    matriz_final.groupby('localidad')['estrato_predominante'].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

# Fusión con Transmilenio
matriz_final = matriz_final.merge(conteo_estaciones, on=['barrio'], how='left')
matriz_final['num_estaciones_transmilenio'] = matriz_final['num_estaciones_transmilenio'].fillna(0).astype(int)

# Fusión con Airbnb
matriz_final = matriz_final.merge(listings, on=['barrio', 'localidad'], how='left')
matriz_final['num_listings_airbnb'] = matriz_final['num_listings_airbnb'].fillna(0)

# Manejo de nulos en CAGR si un barrio no tiene serie completa
matriz_final['cagr_15_26'] = matriz_final['cagr_15_26'].fillna(0)

# Corrección metodológica: densidad de listings for km2 y no conteo absoluto
matriz_final['densidad_airbnb_km2'] = matriz_final['num_listings_airbnb'] / matriz_final['area_km2']

Posteriormente, se depuran los registros huérfanos que presentan anomalías inmobiliarias críticas, se audita el estado general de la información, se ordenan las columnas analíticas de acuerdo con la estructura definitiva del modelo y se exporta la matriz unificada final en formatos tabular (CSV) y geográfico (GeoPackage).

In [134]:
matriz_final = matriz_final[
    ~(
        matriz_final['v_ref_2015'].isnull() | (matriz_final['v_ref_2015'] == 0) |
        matriz_final['v_ref_2026'].isnull() | (matriz_final['v_ref_2026'] == 0)
    )
].copy()

print(f"Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: {matriz_final.shape}")

auditar_datos_inmobiliarios(matriz_final)

columnas_ordenadas = ['localidad', 'barrio', 'estrato_predominante', 'area_km2', 'v_ref_2015', 
                      'v_ref_2026', 'cagr_15_26', 'delitos_2018', 'delitos_2026', 'cagr_delitos_18_26', 
                      'num_estaciones_transmilenio', 'num_listings_airbnb', 'densidad_airbnb_km2', 'geometry'
]

matriz_final = matriz_final[columnas_ordenadas]
matriz_final = gpd.GeoDataFrame(matriz_final, geometry='geometry')

# Exportación de archivos finales ordenados
matriz_final.to_csv('matriz_barrios_unificada.csv', index=False)
matriz_final.to_file('matriz_barrios_unificada.gpkg', driver='GPKG')

Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: (1098, 14)

1. Conteo de valores nulos (NaN) por columna:
v_ref_2015    0
v_ref_2026    0
dtype: int64

2. Conteo de valores en cero (0) por columna:
v_ref_2015    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 0 NaNs | 0 ceros (Total anomalías: 0)

4. Total de registros afectados con anomalías entre 2015 - 2026: 0

5. ¡No se encontraron registros críticos entre 2015 - 2026!


In [135]:
matriz_final.head()

,localidad,barrio,estrato_predominante,area_km2,v_ref_2015,v_ref_2026,cagr_15_26,delitos_2018,delitos_2026,cagr_delitos_18_26,num_estaciones_transmilenio,num_listings_airbnb,densidad_airbnb_km2,geometry
0,USME,EL BOSQUE,1.0,0.055879,290000.0,660000.0,0.077625,6852.0,5761.0,-0.021445,0,0.0,0.000000,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525..."
1,SUBA,BILBAO,2.0,0.295314,750000.0,2050000.0,0.095719,22038.0,17403.0,-0.029084,0,0.0,0.000000,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,3.0,0.476473,1000000.0,2640000.0,0.092264,24725.0,17188.0,-0.044433,0,7.0,14.691286,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657..."
3,CHAPINERO,CONCEPCION NORTE,3.0,0.140259,1400000.0,4380000.0,0.109255,13995.0,9476.0,-0.047573,1,0.0,0.000000,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727..."
4,PUENTE ARANDA,COLON,3.0,0.285947,1300000.0,3170000.0,0.084407,9632.0,9405.0,-0.002977,0,16.0,55.954334,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043..."
